<a href="https://colab.research.google.com/github/Muskan-Kandel/aes/blob/muskan%2Fdata-preprocessing/AES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install nltk transformers torch -q

# Install required libraries
import re
import json
import nltk
import torch
import pandas as pd
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

#punkt and punkt_tab is used for tokenization
#(breaking paragraph into a list of words)
nltk.download('punkt')
nltk.download('punkt_tab')


Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [3]:
asap_raw = pd.read_csv('/content/drive/MyDrive/aes/training_set_rel3.csv', encoding='latin1')
asap_raw.shape

(12978, 28)

In [4]:
asap_raw.head()


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4.0,4.0,NaN,8.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5.0,4.0,NaN,9.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4.0,3.0,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5.0,5.0,NaN,10.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4.0,4.0,NaN,8.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
prompt3 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-3.csv', encoding='latin1')
prompt3.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [6]:
prompt4 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-4.csv', encoding='latin1')
prompt4.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,8863,0,0,0,0
1,8864,0,0,0,0
2,8865,3,3,3,3
3,8866,3,3,2,2
4,8867,2,2,2,2


In [7]:
prompt5 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-5.csv', encoding='latin1')
prompt5.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,11827,2,2,2,2
1,11828,3,3,3,3
2,11829,2,2,2,2
3,11830,1,0,1,1
4,11831,4,4,4,4


In [8]:
prompt6 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-6.csv', encoding='latin1')
prompt6.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,15491,4,3,2,2
1,15972,1,1,3,2
2,16109,4,4,3,3
3,16228,2,1,4,3
4,16352,3,3,4,4


In [9]:
filtered= asap_raw[asap_raw['essay_set'].isin([3,4,5,6])]
print("Total rows in prompt 3-6 is", len(filtered))
print("Columns : ", list(filtered.columns))
print(f"Shape: {filtered.shape}")
print("Per prompt : ", filtered['essay_set'].value_counts().sort_index().to_dict())
for i in [3,4,5,6]:
  p = pd.read_csv(f'/content/drive/MyDrive/aes/Prompt-{i}.csv')
  print(f'prompt-{i}.csv rows: {len(p)}, columns: {list(p.columns)}')

  #check for essay_id overlap in asap dataset and prompt 3-6 datasets

  overlap = filtered[filtered['essay_set'] == i]['essay_id'].isin(p['Essay ID']).sum()
  print(f'Matching essay_ids: {overlap}/{len(p)}')


Total rows in prompt 3-6 is 7103
Columns :  ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Shape: (7103, 28)
Per prompt :  {3: 1726, 4: 1772, 5: 1805, 6: 1800}
prompt-3.csv rows: 1726, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1726/1726
prompt-4.csv rows: 1772, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1772/1772
prompt-5.csv rows: 1805, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1805/1805
prompt-6.csv rows


This shows for every essay in filtered asap_raw data (Sets 3, 4, 5, and 6), there is a corresponding row in the Prompt-X.csv files.

In [10]:
filtered_clean = filtered[['essay_id', 'essay_set', 'essay', 'domain1_score']].copy()
trait_dfs = []
for i in [3, 4, 5, 6]:
    prompt_df = pd.read_csv(f'/content/drive/MyDrive/aes/Prompt-{i}.csv')

    # Standardize column names if necessary (e.g., 'Essay ID' to 'essay_id')
    prompt_df = prompt_df.rename(columns={'Essay ID': 'essay_id'})
    trait_dfs.append(prompt_df)

all_traits = pd.concat(trait_dfs, ignore_index = True)

#merge on essay_id
asap_plus_plus = filtered_clean.merge(all_traits, on='essay_id', how='inner')

#clean dataset
asap_plus_plus.dropna(subset=['essay', 'domain1_score'], inplace=True)
asap_plus_plus.drop_duplicates(subset=['essay'], inplace=True)
asap_plus_plus.reset_index(drop=True, inplace=True)

#check final dataset
print("Final columns:", list(asap_plus_plus.columns))
print("Final shape:", asap_plus_plus.shape)
print("Per prompt:\n", asap_plus_plus['essay_set'].value_counts().sort_index())
print("\nSample:")
asap_plus_plus.head(3)


Final columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Final shape: (7098, 8)
Per prompt:
 essay_set
3    1724
4    1769
5    1805
6    1800
Name: count, dtype: int64

Sample:


,essay_id,essay_set,essay,domain1_score,Content,Prompt Adherence,Language,Narrativity
0,5978,3,The features of the setting affect the cyclist...,1.0,0,0,1,1
1,5979,3,The features of the setting affected the cycli...,2.0,3,3,2,2
2,5980,3,Everyone travels to unfamiliar places. Sometim...,1.0,2,2,2,1


# **Score Normalization (Scale 0-10)**

Each prompt in ASAP++ has a different maximum score: Prompts 3 and 4 go up to **3** and Prompts 5 and 6 go up to **4**. This inconsistency can confuse the model, making it think a score of 3 means different things depending on the prompt.

To fix this, we scale all scores to a unified **0 to 10** range using:

$$Score_{norm} = \left( \frac{Score_{raw}}{S_{max}} \right) \times 10$$

| Prompt | Max Score | After Normalization |
| :--- | :--- | :--- |
| 3 | 3 | 10 |
| 4 | 3 | 10 |
| 5 | 4 | 10 |
| 6 | 4 | 10 |

This applies to all 5 score columns — `domain1_score`, `Content`, `Prompt Adherence`, `Language`, and `Narrativity`.

In [11]:
# Defining max scores per prompt
max_score_map = {3: 3, 4: 3, 5: 4, 6: 4}

# Trait columns to normalize
trait_cols = ['domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']

def normalize_scores(df):
    df = df.copy()
    for col in trait_cols:
        df[col] = df.apply(
            lambda row: round((row[col] / max_score_map[row['essay_set']]) * 10, 4),
            axis=1
        )
    return df

asap_plus_plus = normalize_scores(asap_plus_plus)

print("Scores normalized to 0-10 scale")
print("Sample scores after normalization:")
print(asap_plus_plus[['essay_set', 'domain1_score', 'Content', 'Prompt Adherence',
                        'Language', 'Narrativity']].head(5))


Scores normalized to 0-10 scale
Sample scores after normalization:
   essay_set  domain1_score  Content  Prompt Adherence  Language  Narrativity
0          3         3.3333   0.0000            0.0000    3.3333       3.3333
1          3         6.6667  10.0000           10.0000    6.6667       6.6667
2          3         3.3333   6.6667            6.6667    6.6667       3.3333
3          3         3.3333   0.0000            3.3333    0.0000       0.0000
4          3         6.6667   3.3333            3.3333    3.3333       3.3333


# **Text Normalization**

Raw essays from the ASAP dataset are messy. They contain encoding garbage like `ô` and `ö`, extra spaces, newlines and non-ASCII characters. We clean all of this so the model gets consistent, readable text.

**Steps:**
1. Fix encoding artifacts (`ô` -> `"`, `æ` ->`'`)
2. Remove HTML tags and URLs
3. Normalize whitespace and newlines
4. Lowercase everything
5. Remove non-ASCII characters
6. Fix punctuation spacing
7. Restore ASAP tokens back into the cleaned text


In [12]:
ASAP_TOKEN_PATTERN = re.compile(
    r'@(CAPS\d*|NUM\d*|PERSON\d*|ORGANIZATION\d*|LOCATION\d*|'
    r'DATE\d*|MONTH\d*|TIME\d*|MONEY\d*|PERCENT\d*|DR\d*|STATE\d*|CITY\d*)',
    re.IGNORECASE
)

def normalize_essay(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # Protect ASAP tokens
    asap_tokens = {}
    counter = [0]
    def replace_asap(match):
        key = f"ASAPTOKEN{counter[0]}"
        asap_tokens[key] = match.group(0).upper()
        counter[0] += 1
        return key
    text = ASAP_TOKEN_PATTERN.sub(replace_asap, text)

    # Fix encoding artifacts
    fixes = {
        'ô': '"', 'ö': '"', 'æ': "'", 'Æ': "'", 'ø': 'o',
        '\x93': '"', '\x94': '"', '\x91': "'", '\x92': "'",
        '\x85': '...', '\x96': '-', '\x97': '-',
        '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    }
    for bad, good in fixes.items():
        text = text.replace(bad, good)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Normalize whitespace
    text = re.sub(r'[\t\n\r]+', ' ', text)

    # Remove non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Fix punctuation spacing
    text = re.sub(r'\s([.,!?])', r'\1', text)
    text = re.sub(r'([.,!?])([^\s\d])', r'\1 \2', text)

    # Collapse spaces
    text = re.sub(r' +', ' ', text).strip()

    # Restore ASAP tokens
    for key, original in asap_tokens.items():
        text = text.replace(key, original)

    return text

# Apply and save to new column
print("Normalizing essay text")
asap_plus_plus['essay_normalized'] = asap_plus_plus['essay'].apply(normalize_essay)
print("Text normalized")


print("Columns now:", list(asap_plus_plus.columns))
print("\nBEFORE:", asap_plus_plus['essay'].iloc[1000][:200])
print("\nAFTER: ", asap_plus_plus['essay_normalized'].iloc[1000][:200])

Normalizing essay text
Text normalized
Columns now: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity', 'essay_normalized']

BEFORE: In the story ôRough Road Ahead: Do not Exceed posted speed limit ôthe cyclist has some very rough obstacles on his way to Yosemite National park. With his lack of water and diredness he runs into a bi

AFTER:  In the story "Rough Road Ahead: Do not Exceed posted speed limit "the cyclist has some very rough obstacles on his way to Yosemite National park. With his lack of water and diredness he runs into a bi


# **RUBRIC JSON MAPPING**

**Problem:** Mistral doesn't know what it's supposed to grade

**Solution:** For each essay, prepend a rubric that tells the
model exactly what scoring criteria to use for that prompt

Each prompt has different writing requirements:

Prompt 3 and 4 -> Source-dependent (reading comprehension and evidence usage)

Prompt 5 and 6 -> Persuasive essays (argument, reasoning and organization)

**Steps:**

1. Create a rubric mapping dictionary that links each essay_set (prompt ID) to its grading rubric.

2. Save the rubric mapping as a JSON file for reference and reproducibility.

3. For each essay, retrieve the rubric corresponding to its prompt ID.

4. Combine the rubric with the cleaned essay text.

5. Format the input using Mistral’s instruction format

**Result:**

Each essay is paired with the correct grading rubric, ensuring the model evaluates the essay according to the appropriate scoring criteria before generating a score.

In [13]:
rubric_mapping = {
    3: (
        "You are an expert essay grader. Grade the following student essay written in response to "
        "a source-dependent reading task. The student must demonstrate understanding of the source text, "
        "use relevant evidence, and show comprehension of the passage's key ideas. "
        "Evaluate on: Content (relevance and evidence), Prompt Adherence (stays on task), "
        "Language (grammar and spelling), Narrativity (logical flow of ideas). "
        "All scores are on a scale of 0 to 10."
    ),
    4: (
        "You are an expert essay grader. Grade the following student essay written in response to "
        "a source-dependent reading task. The student must interpret and respond to a specific reading passage, "
        "demonstrating critical thinking and textual evidence. "
        "Evaluate on: Content (depth of analysis), Prompt Adherence (addresses the specific passage), "
        "Language (grammar and spelling), Narrativity (coherent progression of ideas). "
        "All scores are on a scale of 0 to 10."
    ),
    5: (
        "You are an expert essay grader. Grade the following student persuasive essay. "
        "The student must present a clear argument, support it with reasoning and evidence, "
        "and maintain a consistent persuasive voice. "
        "Evaluate on: Content (strength of argument), Prompt Adherence (addresses the prompt), "
        "Language (grammar, word choice, sentence fluency), Narrativity (organization and flow). "
        "All scores are on a scale of 0 to 10."
    ),
    6: (
        "You are an expert essay grader. Grade the following student persuasive essay. "
        "The student must demonstrate effective use of writing conventions, sentence variety, "
        "and a well-organized argument with a clear position. "
        "Evaluate on: Content (quality of reasoning), Prompt Adherence (clear position on topic), "
        "Language (conventions, grammar, spelling), Narrativity (structure and idea progression). "
        "All scores are on a scale of 0 to 10."
    ),
}

# Saving rubric as a JSON file
with open('rubric_mapping.json', 'w') as f:
    json.dump(rubric_mapping, f, indent=2)
print(" rubric_mapping.json saved")

# Build the full Mistral input string for each essay
# Format: [INST] {rubric} [/INST] {essay}
# This is the official Mistral instruction format
def build_mistral_input(row):
    rubric = rubric_mapping[row['essay_set']]
    essay  = row['essay_normalized']
    return f"[INST] {rubric} [/INST] {essay}"

asap_plus_plus['mistral_input'] = asap_plus_plus.apply(build_mistral_input, axis=1)

print("\nSample mistral_input:")
print(asap_plus_plus['mistral_input'].iloc[0][:400])

 rubric_mapping.json saved

Sample mistral_input:
[INST] You are an expert essay grader. Grade the following student essay written in response to a source-dependent reading task. The student must demonstrate understanding of the source text, use relevant evidence, and show comprehension of the passage's key ideas. Evaluate on: Content (relevance and evidence), Prompt Adherence (stays on task), Language (grammar and spelling), Narrativity (logical


# **BPE TOKENIZATION**

**Problem**: Mistral only understands numbers, not text

**Solution** : Use Mistral's own tokenizer (BPE — Byte Pair Encoding) to convert text into input_ids and attention_mask


**What is BPE?**

Instead of splitting by words, BPE splits by common subword units.

This handles rare/misspelled words without "unknown token" errors


**What are input_ids?**

A list of integers —> each number = one token in Mistral's vocabulary
e.g. "the cat" -> [1, 28705, 5255] (Mistral's internal IDs)


**What is attention_mask?**

1 = real token, 0 = padding
Since essays have different lengths, shorter ones get padded
The mask tells Mistral which tokens to actually pay attention to


In [14]:
print("\nLoading Mistral tokenizer...")
# Using Mistral-7B-v0.1 tokenizer from HuggingFace
# This requires being logged in: huggingface-cli login
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

# Mistral tokenizer has no default pad token but batching padding, so, set it to eos_token
tokenizer.pad_token = tokenizer.eos_token

# Max token length 512 is safe for essays of 150-250 words
# Going higher uses more GPU memory
MAX_LENGTH = 512

def tokenize_batch(texts):

    encoded = tokenizer(
        texts,
        padding='max_length',       # pad all sequences to MAX_LENGTH
        truncation=True,            # cut sequences longer than MAX_LENGTH
        max_length=MAX_LENGTH,
        return_tensors='pt'         # return PyTorch tensors instead of python lists
    )
    return encoded['input_ids'], encoded['attention_mask']

# Tokenize in batches of 32 to avoid memory issues
BATCH_SIZE = 32
all_input_ids      = []
all_attention_masks = []

texts = asap_plus_plus['mistral_input'].tolist()
total  = len(texts)

print(f"Tokenizing {total} essays in batches of {BATCH_SIZE}...")
for start in range(0, total, BATCH_SIZE):
    batch = texts[start : start + BATCH_SIZE]
    input_ids, attention_mask = tokenize_batch(batch)
    all_input_ids.append(input_ids)
    all_attention_masks.append(attention_mask)

    # Print progress for every 20 batches
    if (start // BATCH_SIZE) % 20 == 0:
        print(f"  Processed {min(start + BATCH_SIZE, total)}/{total}")

# Stack all batches into final tensors
final_input_ids       = torch.cat(all_input_ids, dim=0)
final_attention_masks = torch.cat(all_attention_masks, dim=0)

print(f"\n Tokenization complete!")
print(f"   input_ids shape:      {final_input_ids.shape}")
print(f"   attention_mask shape: {final_attention_masks.shape}")

# Decoded example to verify it looks right
print("\nDecoded sample (first essay, first 100 tokens):")
print(tokenizer.decode(final_input_ids[0][:100], skip_special_tokens=False))


Loading Mistral tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizing 7098 essays in batches of 32...
  Processed 32/7098
  Processed 672/7098
  Processed 1312/7098
  Processed 1952/7098
  Processed 2592/7098
  Processed 3232/7098
  Processed 3872/7098
  Processed 4512/7098
  Processed 5152/7098
  Processed 5792/7098
  Processed 6432/7098
  Processed 7072/7098

 Tokenization complete!
   input_ids shape:      torch.Size([7098, 512])
   attention_mask shape: torch.Size([7098, 512])

Decoded sample (first essay, first 100 tokens):
<s> [INST] You are an expert essay grader. Grade the following student essay written in response to a source-dependent reading task. The student must demonstrate understanding of the source text, use relevant evidence, and show comprehension of the passage's key ideas. Evaluate on: Content (relevance and evidence), Prompt Adherence (stays on task), Language (grammar and spelling), Narrativity (logical flow of ideas). All scores are on a


In [15]:
# SAVING

torch.save({
    'input_ids': final_input_ids,
    'attention_mask': final_attention_masks,
}, '/content/drive/MyDrive/aes/mistral_input_tensors.pt')

# Save processed dataframe
asap_plus_plus.to_csv('/content/drive/MyDrive/aes/asap_plus_plus_processed.csv', index=False)

print("All saved to google drive")

All saved to google drive
